# Conformal prediction

Paper-facing tables for the 90% conformal prediction experiments. The analysis uses the linear probe, excludes TENX143 and Hibou-L, retains only LEMON ViT-S/8 under the label LEMON, and labels the CellBridge all/adapted representation simply as CellBridge.

The first table compares representations under branch split, the proposed conformal construction. The second fixes the CellBridge representation and compares the proposed branch split with global split, global graph, and class split. HEST (ID) and all OOD benchmarks appear in the same table.

In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

WORKSPACE = Path(os.environ.get(
    "CELLBRIDGE_WORKSPACE",
    "/lustre/scratch126/cellgen/lotfollahi/so25",
))
RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_INFERENCE_RESULTS",
    WORKSPACE / "outputs/reproducibility/inference",
))

PROBE = "linear"
EXCLUDED_HEST_SAMPLES = {"TENX143"}
METHOD_COMPARISON_CP = "branch_split"

METHOD_GROUPS = [
    ("Patch-level FMs", ["UNI2-h", "Virchow2"]),
    ("Cell-level FMs", ["VOLTA", "LEMON"]),
    ("Multi-modal FMs", ["PAST", "SpatialWhisperer"]),
    ("Ours", ["CellBridge"]),
]
METHOD_ORDER = [
    method
    for _, methods in METHOD_GROUPS
    for method in methods
]

ABLATION_ORDER = ["split", "graph", "class_split", "branch_split"]
CONFORMAL_LABELS = {
    "split": "Global split",
    "graph": "Global graph",
    "class_split": "Class split",
    "branch_split": "Branch split (ours)",
}

BENCHMARK_ORDER = [
    "HEST",
    "Lizard",
    "NuCLS",
    "PanNuke",
    "PhenoCell (all)",
    "PhenoCell (immune)",
]
BENCHMARK_LABELS = {
    "HEST": "HEST (ID)",
    "Lizard": "Lizard (OOD)",
    "NuCLS": "NuCLS (OOD)",
    "PanNuke": "PanNuke (OOD)",
    "PhenoCell (all)": "PhenoCell all (OOD)",
    "PhenoCell (immune)": "PhenoCell immune (OOD)",
}

TABLE_METRICS = [
    ("marginal_coverage", "Marginal coverage"),
    ("mean_set_size", "Mean set size (↓)"),
    ("singleton_rate", "Singleton rate (↑)"),
]
LATEX_METRIC_LABELS = {
    "marginal_coverage": r"Marginal coverage",
    "mean_set_size": r"Mean set size ($\downarrow$)",
    "singleton_rate": r"Singleton rate ($\uparrow$)",
}

ROW_COLOUR = "gray!15"
OURS_BACKGROUND = "#E6E6E6"

## Load and aggregate the conformal metrics

HEST entries are means across the 14 retained tissue sections; each OOD entry comes from its fixed held-out test split. Marginal coverage and singleton rate are reported as percentages. Mean set size is the mean number of labels retained per prediction set.

In [2]:
BASELINE_LABELS = {
    "uni": "UNI2-h",
    "virchow2": "Virchow2",
    "lemon": "LEMON",
    "past": "PAST",
    "volta": "VOLTA",
    "spatialwhisperer": "SpatialWhisperer",
}


def paper_method(method, version, arm):
    if method == "cellbridge" or arm.startswith("lora_"):
        size = version.removeprefix("cells_")
        size = size.upper() if size != "all" else "all"
        representation = arm.removeprefix("lora_")
        if size == "all" and representation == "adapted":
            return "CellBridge"
        return "CellBridge {} ({})".format(size, representation)
    return BASELINE_LABELS.get(method, method)


def benchmark_label(dataset, task):
    if dataset == "phenocell":
        return "PhenoCell ({})".format(task)
    return {
        "hest": "HEST",
        "lizard": "Lizard",
        "nucls": "NuCLS",
        "pannuke": "PanNuke",
    }[dataset]


def load_conformal_metrics(root):
    paths = list(root.glob(
        "*/*/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    paths += list(root.glob(
        "*/*/hest/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    tables = []
    required = {
        "method",
        "arm",
        "task",
        "probe",
        "conformal_method",
        "target_coverage",
        "marginal_coverage",
        "mean_prediction_set_size",
        "singleton_rate",
    }
    for path in sorted(paths):
        parts = path.relative_to(root).parts
        method, version, dataset = parts[:3]
        if dataset == "hest":
            sample_id = parts[3]
            arm, task, probe = parts[5:8]
        else:
            sample_id = pd.NA
            arm, task, probe = parts[4:7]

        table = pd.read_csv(path)
        missing = required.difference(table.columns)
        if missing:
            raise ValueError(
                "{} is missing columns: {}".format(path, sorted(missing))
            )
        table["version"] = version
        table["sample_id"] = sample_id
        table["paper_method"] = paper_method(method, version, arm)
        table["benchmark"] = benchmark_label(dataset, task)
        table["result_dir"] = str(path.parent)
        tables.append(table)

    if not tables:
        raise FileNotFoundError(
            "No conformal metrics found below {}".format(root)
        )

    table = pd.concat(tables, ignore_index=True)
    table = table.loc[table["method"] != "hibou_l"]
    table = table.loc[
        ~((table["method"] == "lemon") & (table["version"] != "vits8"))
    ]
    table = table.loc[
        ~table["sample_id"].astype(str).isin(EXCLUDED_HEST_SAMPLES)
    ]
    return table.reset_index(drop=True)


def aggregate_points(table):
    return (
        table.groupby(
            ["paper_method", "benchmark", "probe", "conformal_method"],
            as_index=False,
        )
        .agg(
            marginal_coverage=("marginal_coverage", "mean"),
            mean_set_size=("mean_prediction_set_size", "mean"),
            singleton_rate=("singleton_rate", "mean"),
            target_coverage=("target_coverage", "mean"),
            n_hest_sections=("sample_id", "count"),
        )
    )


conformal = load_conformal_metrics(RESULTS_ROOT)
points = aggregate_points(conformal)

target_coverages = conformal["target_coverage"].dropna().unique()
if len(target_coverages) != 1:
    raise ValueError(
        "Expected one nominal coverage target, found {}".format(
            sorted(target_coverages)
        )
    )
NOMINAL_COVERAGE = float(target_coverages[0])

print(
    "Loaded {:,} conformal rows at {:.0%} nominal coverage; "
    "probe = {}; excluded HEST = {}".format(
        len(conformal),
        NOMINAL_COVERAGE,
        PROBE,
        sorted(EXCLUDED_HEST_SAMPLES),
    )
)

Loaded 4,256 conformal rows at 90% nominal coverage; probe = linear; excluded HEST = ['TENX143']


## Representation comparison under branch split

This table compares all baseline representations with CellBridge using branch split and the linear probe. Methods are grouped by representation family, with the three metrics stacked as rows beneath each method. Dataset columns are grouped into ID and OOD. The three gray CellBridge rows identify our proposed representation and conformal construction.


In [3]:
def _format_metric(value, metric):
    if pd.isna(value):
        return "--"
    if metric in {"marginal_coverage", "singleton_rate"}:
        return "{:.1f}".format(100 * value)
    return "{:.2f}".format(value)


TABLE_BENCHMARK_LABELS = {
    "HEST": "HEST",
    "Lizard": "Lizard",
    "NuCLS": "NuCLS",
    "PanNuke": "PanNuke",
    "PhenoCell (all)": "PhenoCell all",
    "PhenoCell (immune)": "PhenoCell immune",
}
LATEX_BENCHMARK_LABELS = {
    "PhenoCell all": r"PhenoCell\textsubscript{all}",
    "PhenoCell immune": r"PhenoCell\textsubscript{imm}",
}
BENCHMARK_SPLITS = {
    benchmark: "ID" if benchmark == "HEST" else "OOD"
    for benchmark in BENCHMARK_ORDER
}


def build_rowwise_table(
    rows,
    row_key,
    row_order,
    row_heading,
    row_labels=None,
):
    duplicate = rows.duplicated([row_key, "benchmark"], keep=False)
    if duplicate.any():
        raise ValueError(
            "Duplicate table entries:\n{}".format(
                rows.loc[
                    duplicate,
                    [row_key, "benchmark"],
                ].to_string(index=False)
            )
        )

    observed = set(
        rows[[row_key, "benchmark"]].itertuples(index=False, name=None)
    )
    missing = [
        (row, benchmark)
        for row in row_order
        for benchmark in BENCHMARK_ORDER
        if (row, benchmark) not in observed
    ]
    if missing:
        raise ValueError("Missing table entries: {}".format(missing))

    metric_keys = [metric for metric, _ in TABLE_METRICS]
    raw = rows.pivot(
        index=row_key,
        columns="benchmark",
        values=metric_keys,
    )
    labels = row_labels or {}
    values = []
    index = []
    for row in row_order:
        row_label = labels.get(row, row)
        for metric, metric_label in TABLE_METRICS:
            index.append((row_label, metric_label))
            values.append([
                _format_metric(raw.loc[row, (metric, benchmark)], metric)
                for benchmark in BENCHMARK_ORDER
            ])

    columns = pd.MultiIndex.from_tuples(
        [
            (
                BENCHMARK_SPLITS[benchmark],
                TABLE_BENCHMARK_LABELS[benchmark],
            )
            for benchmark in BENCHMARK_ORDER
        ],
        names=["Split", "Dataset"],
    )
    return pd.DataFrame(
        values,
        index=pd.MultiIndex.from_tuples(
            index,
            names=[row_heading, "Metric"],
        ),
        columns=columns,
    )


def rank_table_cells(rows, row_key, row_order, row_labels=None):
    labels = row_labels or {}
    lookup = rows.set_index([row_key, "benchmark"])
    emphasis = {"best": set(), "second": set()}

    for metric, metric_label in TABLE_METRICS:
        for benchmark in BENCHMARK_ORDER:
            candidates = []
            for row in row_order:
                value = lookup.loc[(row, benchmark), metric]
                if pd.isna(value):
                    continue
                displayed_value = float(_format_metric(value, metric))
                if metric == "marginal_coverage":
                    score = round(abs(displayed_value - 90.0), 10)
                elif metric == "mean_set_size":
                    score = displayed_value
                else:
                    score = -displayed_value
                candidates.append((row, score))

            score_levels = sorted({score for _, score in candidates})
            for rank, score_level in zip(
                ("best", "second"),
                score_levels[:2],
            ):
                for row, score in candidates:
                    if score == score_level:
                        emphasis[rank].add((
                            labels.get(row, row),
                            metric_label,
                            TABLE_BENCHMARK_LABELS[benchmark],
                        ))
    return emphasis


def display_table(table, emphasis, shaded_rows, label_level):
    def shade(row):
        row_label = row.name[label_level]
        colour = (
            "background-color: {}".format(OURS_BACKGROUND)
            if row_label in shaded_rows
            else ""
        )
        return [colour] * len(row)

    def emphasize(data):
        styles = pd.DataFrame("", index=data.index, columns=data.columns)
        for index in data.index:
            row_label = index[label_level]
            metric_label = index[-1]
            for column in data.columns:
                key = (row_label, metric_label, column[-1])
                if key in emphasis["best"]:
                    styles.at[index, column] = "font-weight: bold"
                elif key in emphasis["second"]:
                    styles.at[index, column] = "text-decoration: underline"
        return styles

    display(
        table.style
        .apply(shade, axis=1)
        .apply(emphasize, axis=None)
        .set_properties(**{"text-align": "center"})
        .set_table_styles([
            {
                "selector": "th.row_heading.level{}".format(label_level),
                "props": [
                    ("text-align", "center"),
                    ("vertical-align", "middle"),
                ],
            }
        ])
    )


def _header_segments(columns, level):
    segments = []
    start = 0
    for position in range(1, len(columns) + 1):
        if (
            position == len(columns)
            or columns[position][level] != columns[start][level]
        ):
            segments.append((columns[start][level], start, position - 1))
            start = position
    return segments


def emit_latex_table(
    table,
    caption,
    label,
    emphasis,
    shaded_rows,
    label_level,
    row_heading,
    grouped=False,
):
    total_columns = 2 + len(table.columns)
    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\setlength{\tabcolsep}{4pt}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll" + "c" * len(table.columns) + "}",
        r"\toprule",
    ]

    split_headers = []
    split_rules = []
    for split, start, end in _header_segments(table.columns, level=0):
        split_headers.append(
            r"\multicolumn{"
            + str(end - start + 1)
            + r"}{c}{\textbf{"
            + str(split)
            + "}}"
        )
        split_rules.append(
            r"\cmidrule(lr){"
            + "{}-{}".format(start + 3, end + 3)
            + "}"
        )
    lines.append(" & & " + " & ".join(split_headers) + r" \\")
    lines.append(" ".join(split_rules))
    lines.append(
        row_heading
        + " & Metric & "
        + " & ".join(
            LATEX_BENCHMARK_LABELS.get(column[-1], column[-1])
            for column in table.columns
        )
        + r" \\"
    )
    lines.append(r"\midrule")

    latex_metric_labels = {
        display_label: LATEX_METRIC_LABELS[metric]
        for metric, display_label in TABLE_METRICS
    }
    previous_group = None
    previous_row = None
    have_rows = False
    for index, row in table.iterrows():
        group = index[0] if grouped else None
        row_label = index[label_level]
        metric_label = index[-1]

        if grouped and group != previous_group:
            if have_rows:
                lines.append(r"\midrule")
            lines.append(
                r"\multicolumn{"
                + str(total_columns)
                + r"}{l}{\textbf{"
                + str(group)
                + r"}} \\"
            )
            lines.append(r"\addlinespace")
            previous_row = None
        elif row_label != previous_row and have_rows:
            lines.append(r"\addlinespace")

        prefix = (
            r"\rowcolor{" + ROW_COLOUR + "} "
            if row_label in shaded_rows
            else ""
        )
        is_first_metric = row_label != previous_row
        is_last_metric = metric_label == TABLE_METRICS[-1][1]
        if row_label in shaded_rows:
            shown_row_label = (
                r"\multirow{-"
                + str(len(TABLE_METRICS))
                + r"}{*}{"
                + str(row_label)
                + "}"
                if is_last_metric
                else ""
            )
        else:
            shown_row_label = (
                r"\multirow{"
                + str(len(TABLE_METRICS))
                + r"}{*}{"
                + str(row_label)
                + "}"
                if is_first_metric
                else ""
            )
        formatted_values = []
        for column, value in row.items():
            key = (row_label, metric_label, column[-1])
            shown_value = str(value)
            if key in emphasis["best"]:
                shown_value = r"\textbf{" + shown_value + "}"
            elif key in emphasis["second"]:
                shown_value = r"\underline{" + shown_value + "}"
            formatted_values.append(shown_value)
        lines.append(
            prefix
            + shown_row_label
            + " & "
            + latex_metric_labels[metric_label]
            + " & "
            + " & ".join(formatted_values)
            + r" \\"
        )
        previous_group = group
        previous_row = row_label
        have_rows = True

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"}",
        r"\end{table*}",
    ])
    print("\n".join(lines))


main_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["conformal_method"] == METHOD_COMPARISON_CP)
    & points["paper_method"].isin(METHOD_ORDER)
].copy()
main_table = build_rowwise_table(
    main_rows,
    row_key="paper_method",
    row_order=METHOD_ORDER,
    row_heading="Method",
)
family_by_method = {
    method: family
    for family, methods in METHOD_GROUPS
    for method in methods
}
main_table.index = pd.MultiIndex.from_tuples(
    [
        (family_by_method[method], method, metric)
        for method, metric in main_table.index
    ],
    names=["Family", "Method", "Metric"],
)

main_emphasis = rank_table_cells(
    main_rows,
    row_key="paper_method",
    row_order=METHOD_ORDER,
)
main_shaded_rows = {"CellBridge"}
display_table(
    main_table,
    emphasis=main_emphasis,
    shaded_rows=main_shaded_rows,
    label_level=1,
)
emit_latex_table(
    main_table,
    caption=(
        "Representation comparison under branch split using linear probes. "
        "Marginal coverage and singleton rate are percentages; the nominal "
        "{:.0f}\\% coverage target should be considered together with mean "
        "set size and singleton rate. HEST is ID and all other benchmarks "
        "are OOD. HEST entries average 14 retained sections. Gray rows "
        "identify CellBridge."
    ).format(100 * NOMINAL_COVERAGE),
    label="tab:conformal-representation-comparison",
    emphasis=main_emphasis,
    shaded_rows=main_shaded_rows,
    label_level=1,
    row_heading="Method",
    grouped=True,
)


\begin{table*}[t]
\centering
\caption{Representation comparison under branch split using linear probes. Marginal coverage and singleton rate are percentages; the nominal 90\% coverage target should be considered together with mean set size and singleton rate. HEST is ID and all other benchmarks are OOD. HEST entries average 14 retained sections. Gray rows identify CellBridge.}
\label{tab:conformal-representation-comparison}
\setlength{\tabcolsep}{4pt}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llcccccc}
\toprule
 & & \multicolumn{1}{c}{\textbf{ID}} & \multicolumn{5}{c}{\textbf{OOD}} \\
\cmidrule(lr){3-3} \cmidrule(lr){4-8}
Method & Metric & HEST & Lizard & NuCLS & PanNuke & PhenoCell\textsubscript{all} & PhenoCell\textsubscript{imm} \\
\midrule
\multicolumn{8}{l}{\textbf{Patch-level FMs}} \\
\addlinespace
\multirow{3}{*}{UNI2-h} & Marginal coverage & 89.2 & 90.6 & \textbf{90.0} & 89.7 & 88.0 & \textbf{89.7} \\
 & Mean set size ($\downarrow$) & 3.72 & 2.92 & 2.38 & 1.52 & 4.75 & 2.81 \

## CellBridge conformal-construction ablation

All rows use the same CellBridge representation and linear probe. Each conformal configuration has three metric rows, and dataset columns are grouped into ID and OOD. Global split is the standard marginal conformal baseline, global graph adds graph-based risk control, class split calibrates each label separately, and branch split calibrates the proposed biological branches.

Class split is included because it directly tests whether fine-grained label conditioning is preferable to branch-level conditioning. It can be less stable for labels with limited calibration support, so it should be interpreted alongside mean set size and singleton rate. The three gray rows identify the proposed branch-split configuration.


In [4]:
ablation_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["paper_method"] == "CellBridge")
    & points["conformal_method"].isin(ABLATION_ORDER)
].copy()
ablation_table = build_rowwise_table(
    ablation_rows,
    row_key="conformal_method",
    row_order=ABLATION_ORDER,
    row_heading="Configuration",
    row_labels=CONFORMAL_LABELS,
)

ablation_emphasis = rank_table_cells(
    ablation_rows,
    row_key="conformal_method",
    row_order=ABLATION_ORDER,
    row_labels=CONFORMAL_LABELS,
)
proposed_configuration = {"Branch split (ours)"}
display_table(
    ablation_table,
    emphasis=ablation_emphasis,
    shaded_rows=proposed_configuration,
    label_level=0,
)
emit_latex_table(
    ablation_table,
    caption=(
        "CellBridge conformal-construction ablation using linear probes. "
        "Marginal coverage and singleton rate are percentages; the nominal "
        "{:.0f}\\% coverage target should be considered together with mean "
        "set size and singleton rate. HEST is ID and all other benchmarks "
        "are OOD. Gray rows identify the proposed branch-split construction."
    ).format(100 * NOMINAL_COVERAGE),
    label="tab:conformal-construction-ablation",
    emphasis=ablation_emphasis,
    shaded_rows=proposed_configuration,
    label_level=0,
    row_heading="Configuration",
)


\begin{table*}[t]
\centering
\caption{CellBridge conformal-construction ablation using linear probes. Marginal coverage and singleton rate are percentages; the nominal 90\% coverage target should be considered together with mean set size and singleton rate. HEST is ID and all other benchmarks are OOD. Gray rows identify the proposed branch-split construction.}
\label{tab:conformal-construction-ablation}
\setlength{\tabcolsep}{4pt}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llcccccc}
\toprule
 & & \multicolumn{1}{c}{\textbf{ID}} & \multicolumn{5}{c}{\textbf{OOD}} \\
\cmidrule(lr){3-3} \cmidrule(lr){4-8}
Configuration & Metric & HEST & Lizard & NuCLS & PanNuke & PhenoCell\textsubscript{all} & PhenoCell\textsubscript{imm} \\
\midrule
\multirow{3}{*}{Global split} & Marginal coverage & 89.2 & \underline{90.6} & \underline{90.1} & \textbf{89.7} & \underline{88.5} & \textbf{91.2} \\
 & Mean set size ($\downarrow$) & \textbf{1.83} & \textbf{1.19} & \textbf{1.55} & \textbf{1.18} & \textbf{3.6

## Reading the tables

- Marginal coverage is a validity criterion, with 90% as the target rather than a quantity to maximize. Values below the target indicate undercoverage; values far above it are conservative.
- Within each dataset and metric, the best displayed value is bold and the second-best is underlined. Marginal coverage is ranked by absolute distance to 90.0, mean set size from low to high, and singleton rate from high to low. Ties at the displayed precision receive the same emphasis.
- Mean set size and singleton rate are complementary efficiency measures. Smaller sets and more singleton predictions are desirable only while coverage remains adequate.
- HEST is the ID benchmark and is averaged across the 14 retained tissue sections. Lizard, NuCLS, PanNuke, PhenoCell (all), and PhenoCell (immune) are OOD benchmarks.
- The main table shades the three CellBridge metric rows because they combine our representation with the proposed branch split. The ablation shades the three branch-split metric rows so the proposed construction remains identifiable when the representation is fixed.
